# Stage 04 — Calibration

Derives and applies gas-concentration calibration coefficients for the four WYO-platform
gas analyzers (Picarro, Ultra460, Ultra321, Pico017), using the tank/dilution calibration
sequences logged in `raw/calibration/tank_details.txt`.

**Two independent methods, clearly separated:**

1. **CH4 & C3H8 — tank-anchored, multi-point.** Every instrument (including Picarro —
   no instrument is assumed exempt) is regressed against known tank concentrations.
   Feb 12 is the canonical calibration event (the only one spanning the full 0–57 ppm
   dilution range with all 5 dilutions + NOAA + zero). Feb 3 / Feb 6 are computed too,
   purely as a drift-QC cross-check, not applied.

2. **C2H6 — cross-instrument peak-alignment. FLAGGED, LOWER CONFIDENCE.** The tank data
   has only one certified C2H6 point (NOAA, 1.63 ppb) — far below real plume
   concentrations, so a tank-anchored fit isn't trustworthy for the range that matters.
   Instead, Ultra460 is treated as the C2H6 reference and Pico017/Ultra321 are calibrated
   against it by matching plume peaks across all WYO co-deployment days. This rests on
   Ultra460's own C2H6 being correct, which is *assumed*, not independently verified —
   every C2H6 coefficient in the output is tagged `confidence: "low"` for this reason.

**Output:** `04_calibrated/` is the single, complete Stage 04 directory — everything
that's in Stage 03 shows up here too, so analysis can pull all data from one place
without reaching back into `03_instrument_aligned/`. Two kinds of content:

- **Calibrated** (Picarro/Ultra460/Ultra321/Pico017 `Raw`+`Eng`): gets `*_cal` columns
  and a `cal_coefs_ref` column pointing at `calibration_coefs.json`.
- **Passthrough, unchanged** (Spectra/Spectralite for those same three instruments —
  no concentration columns to calibrate; and GPS/Anem/Sprinter/LGR in full — no tank
  or cross-cal coverage exists for them). No `cal_coefs_ref` column is added to these,
  which is how to tell "passed through as-is" apart from "calibration was attempted".

In [ ]:
import json
import re
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats
from scipy.signal import find_peaks

sys.path.insert(0, str(Path().resolve().parent))
from paths import STAGE_02_DIR, STAGE_03_DIR, STAGE_04_DIR, TANK_DETAILS_PATH, REPO_ROOT
from src.align import load_aligned_series

print('Imports OK')

---
## A — Parse the tank calibration manifest

In [ ]:
def parse_tank_details(filepath):
    """Parse tank_details.txt into tank concentrations and per-date calibration windows.

    Returns
    -------
    tank : dict
        {tank_key: {'CH4_ppm', 'C3H8_ppm', 'C2H6_ppb'}}  (None where not certified)
    windows_by_date : dict
        {'YYYYMMDD': [{'tank_key', 'start', 'end'}, ...]}
    """
    MONTHS = {m: i + 1 for i, m in enumerate([
        'january', 'february', 'march', 'april', 'may', 'june',
        'july', 'august', 'september', 'october', 'november', 'december'])}

    conc_re = re.compile(r'^([\w\s]+):\s*(.*)')
    val_re  = re.compile(r'([\d.]+)\s*(ppm|ppb)\s+(\w+)', re.I)
    date_re = re.compile(r'(\w+)\s+(\d+),\s+(\d{4})')
    win_re  = re.compile(r'(\d{2}:\d{2}:\d{2})\s+to\s+(\d{2}:\d{2}:\d{2})\s*[-\u2013]\s*(.+)')

    def norm_key(raw):
        s = raw.strip().lower()
        if s in ('n2 zero', 'n2_zero'):
            return 'N2_zero'
        if s in ('noaa', 'noaa tank'):
            return 'NOAA'
        m = re.match(r'dilution\s*(\d)', s)
        return f'Dilution{m.group(1)}' if m else raw.strip().replace(' ', '_')

    tank = {'N2_zero': {'CH4_ppm': 0.0, 'C3H8_ppm': 0.0, 'C2H6_ppb': 0.0}}
    windows_by_date = {}
    current_date = None

    for line in Path(filepath).read_text().splitlines():
        line = line.strip()
        if not line:
            continue

        dm = date_re.search(line)
        if dm:
            month = MONTHS[dm.group(1).lower()]
            current_date = f'{dm.group(3)}{month:02d}{int(dm.group(2)):02d}'
            windows_by_date[current_date] = []
            continue

        wm = win_re.match(line)
        if wm and current_date:
            t0, t1, raw_label = wm.groups()
            date_part = f'{current_date[:4]}-{current_date[4:6]}-{current_date[6:]}'
            windows_by_date[current_date].append({
                'tank_key': norm_key(raw_label),
                'start': f'{date_part} {t0}',
                'end':   f'{date_part} {t1}',
            })
            continue

        if current_date is None:
            cm = conc_re.match(line)
            if cm:
                entry = {'CH4_ppm': None, 'C3H8_ppm': None, 'C2H6_ppb': None}
                for val, unit, gas in val_re.findall(cm.group(2)):
                    val, gas = float(val), gas.upper()
                    if gas == 'CH4':
                        entry['CH4_ppm'] = val if unit.lower() == 'ppm' else val / 1000
                    elif gas == 'C3H8':
                        entry['C3H8_ppm'] = val if unit.lower() == 'ppm' else val / 1000
                    elif gas == 'C2H6':
                        entry['C2H6_ppb'] = val * 1000 if unit.lower() == 'ppm' else val
                tank[norm_key(cm.group(1))] = entry

    return tank, windows_by_date


TANK, WINDOWS_BY_DATE = parse_tank_details(TANK_DETAILS_PATH)
CAL_DATE_CANONICAL = '20260212'

print(f'Tank standards: {list(TANK)}\n')
for date, wins in sorted(WINDOWS_BY_DATE.items()):
    tag = '  <- canonical' if date == CAL_DATE_CANONICAL else ''
    print(f'{date}: {len(wins)} windows{tag} — {[w["tank_key"] for w in wins]}')

---
## B — Load Stage 03 aligned data

One concatenated Series per instrument per species, pulled straight from the good
(non-`bad`, non-`bad_timestamp`) Stage 03 output via `load_aligned_series`.

In [ ]:
INSTRUMENTS = {
    'Picarro':  {'dir': 'WYO_picarro',       'subdir': ''},
    'Ultra460': {'dir': 'WYO_aerisultra460',  'subdir': 'Raw'},
    'Ultra321': {'dir': 'LANL_aerisultra321', 'subdir': 'Raw'},
    'Pico017':  {'dir': 'LANL_aerispico017',  'subdir': 'Raw'},
}

def load_full_series(inst, col):
    cfg = INSTRUMENTS[inst]
    return load_aligned_series(STAGE_03_DIR, cfg['dir'], cfg['subdir'], col)

CH4  = {inst: load_full_series(inst, 'CH4_ppm') for inst in INSTRUMENTS}
C3H8 = {'Ultra321': load_full_series('Ultra321', 'C3H8_ppm')}         # only Ultra321 has a C3H8 channel
C2H6 = {
    'Ultra460': load_full_series('Ultra460', 'C2H6_ppb'),
    'Pico017':  load_full_series('Pico017', 'C2H6_ppb'),
    'Ultra321': load_full_series('Ultra321', 'C2H6_ppm') * 1000.0,     # ppm -> ppb, matches the others
}

for inst, s in CH4.items():
    print(f'{inst:10s} CH4   {len(s):>9,} pts   {s.index[0]} -> {s.index[-1]}')
print()
for inst, s in C2H6.items():
    print(f'{inst:10s} C2H6  {len(s):>9,} pts')

---
## C — CH4 & C3H8 tank calibration (multi-point, tank-anchored)

Single multi-point OLS fit per instrument per date — no piecewise low/high split.
A single fit already achieves R² > 0.9997 for every instrument on Feb 12, and the
residual pattern that motivated a piecewise split in the legacy coefficients shows up
*identically* in Picarro (the reference-grade, presumptively-linear instrument), which
means it reflects small imprecision in the dilution manifold's delivered concentrations,
not instrument nonlinearity — so splitting would just overfit that artifact.

In [ ]:
def linreg(x, y):
    """OLS dropping NaN pairs. Returns (slope, intercept, r2, n)."""
    mask = np.isfinite(x) & np.isfinite(y)
    n = int(mask.sum())
    if n < 2 or x[mask].max() == x[mask].min():
        return np.nan, np.nan, np.nan, n
    sl, ic, r, *_ = stats.linregress(x[mask], y[mask])
    return sl, ic, r ** 2, n


def window_stats(series_dict, windows):
    """Per-window mean/std for every instrument's series."""
    rows = []
    for w in windows:
        t0, t1 = pd.Timestamp(w['start'], tz='UTC'), pd.Timestamp(w['end'], tz='UTC')
        row = {'tank_key': w['tank_key']}
        for inst, s in series_dict.items():
            if s is None:
                continue
            sub = s[t0:t1]
            row[f'{inst}_mean'] = sub.mean() if len(sub) else np.nan
            row[f'{inst}_n']    = len(sub)
        rows.append(row)
    return pd.DataFrame(rows)


def fit_species(stats_df, tank, inst, species_key):
    """Multi-point OLS: tank concentration (x) vs instrument window mean (y)."""
    col = f'{inst}_mean'
    if col not in stats_df.columns:
        return None
    x = stats_df['tank_key'].map(lambda k: tank.get(k, {}).get(species_key)).astype(float).values
    y = stats_df[col].astype(float).values
    sl, ic, r2, n = linreg(x, y)
    if n < 2:
        return None
    return {'slope': sl, 'intercept': ic, 'r2': r2, 'n': n}


CH4_STATS  = {d: window_stats(CH4, w)  for d, w in WINDOWS_BY_DATE.items()}
C3H8_STATS = {d: window_stats(C3H8, w) for d, w in WINDOWS_BY_DATE.items()}

CH4_COEFS_BY_DATE  = {d: {inst: fit_species(CH4_STATS[d], TANK, inst, 'CH4_ppm')
                           for inst in INSTRUMENTS} for d in WINDOWS_BY_DATE}
C3H8_COEFS_BY_DATE = {d: {'Ultra321': fit_species(C3H8_STATS[d], TANK, 'Ultra321', 'C3H8_ppm')}
                       for d in WINDOWS_BY_DATE}

CH4_COEFS  = CH4_COEFS_BY_DATE[CAL_DATE_CANONICAL]
C3H8_COEFS = C3H8_COEFS_BY_DATE[CAL_DATE_CANONICAL]

print('CH4 — canonical coefficients (Feb 12):')
for inst, c in CH4_COEFS.items():
    print(f'  {inst:10s}  {c}' if c else f'  {inst:10s}  no data')
print('\nC3H8 — canonical coefficients (Feb 12):')
for inst, c in C3H8_COEFS.items():
    print(f'  {inst:10s}  {c}' if c else f'  {inst:10s}  no data')

In [ ]:
# ── Drift QC — Feb 3 / Feb 6 vs canonical Feb 12 (report only, not applied) ────────
print('CH4 drift QC — slope / intercept / R\u00b2 per date:\n')
for inst in INSTRUMENTS:
    print(f'  {inst}:')
    for date in sorted(WINDOWS_BY_DATE):
        c = CH4_COEFS_BY_DATE[date].get(inst)
        tag = '  <- canonical' if date == CAL_DATE_CANONICAL else ''
        if c:
            print(f'    {date}  slope={c["slope"]:.5f}  intercept={c["intercept"]:+.5f}  '
                  f'R2={c["r2"]:.6f}  n={c["n"]}{tag}')
        else:
            print(f'    {date}  no data (missing NOAA and/or dilution coverage){tag}')
    print()

In [ ]:
# ── Scatter: tank concentration vs instrument reading, all dates, with fit lines ──
def plot_species_scatter(stats_by_date, coefs_by_date, tank, species_key, title):
    fig = go.Figure()
    colors  = {'Picarro': '#1f77b4', 'Ultra460': '#ff7f0e', 'Ultra321': '#2ca02c', 'Pico017': '#d62728'}
    symbols = {'20260203': 'circle', '20260206': 'square', '20260212': 'diamond'}

    for date, stats_df in sorted(stats_by_date.items()):
        for inst in coefs_by_date[date]:
            col = f'{inst}_mean'
            if col not in stats_df.columns:
                continue
            x = stats_df['tank_key'].map(lambda k: tank.get(k, {}).get(species_key)).astype(float)
            y = stats_df[col].astype(float)
            valid = x.notna() & y.notna()
            if not valid.any():
                continue
            fig.add_trace(go.Scatter(
                x=x[valid], y=y[valid], mode='markers',
                marker=dict(color=colors.get(inst, 'gray'), symbol=symbols.get(date, 'circle'), size=9),
                name=f'{inst} — {date}', legendgroup=inst, legendgrouptitle=dict(text=inst),
            ))

    for inst, c in coefs_by_date[CAL_DATE_CANONICAL].items():
        if c is None:
            continue
        xmax = max(v[species_key] for v in tank.values() if v.get(species_key) is not None) * 1.05
        xfit = np.array([0, xmax])
        fig.add_trace(go.Scatter(
            x=xfit, y=c['slope'] * xfit + c['intercept'], mode='lines',
            line=dict(color=colors.get(inst, 'gray'), width=2),
            name=f'{inst} fit (canonical)', legendgroup=inst, showlegend=False,
        ))

    fig.update_layout(title=title, xaxis_title=f'Tank {species_key}', yaxis_title=f'Instrument {species_key}',
                       template='plotly_white', height=480, hovermode='closest')
    fig.show()


plot_species_scatter(CH4_STATS, CH4_COEFS_BY_DATE, TANK, 'CH4_ppm', 'CH4 calibration — all dates')
plot_species_scatter(C3H8_STATS, C3H8_COEFS_BY_DATE, TANK, 'C3H8_ppm', 'C3H8 calibration — Ultra321, all dates')

---
## D — C2H6 cross-instrument peak-alignment  ⚠️ FLAGGED — lower confidence

Not tank-anchored. Ultra460 is treated as the C2H6 reference; Pico017 and Ultra321 are
calibrated by matching plume peak magnitudes against it, using every WYO co-deployment
date (derived from `routing_manifest.json`, not hardcoded) with the tank calibration
windows excluded. Every coefficient from this section is tagged `confidence: "low"` in
the saved output — see the intro cell for why.

In [ ]:
with open(STAGE_02_DIR / 'routing_manifest.json') as f:
    ROUTING = json.load(f)
WYO_DATES = sorted({f'20{k.split("_")[1]}' for k, v in ROUTING.items() if v == 'WYO'})
print(f'WYO co-deployment dates ({len(WYO_DATES)}): {WYO_DATES}')

# Exclude tank-calibration windows (+-5 min pad) so tank gas doesn't pollute the ambient fit
CAL_WINDOW_PAD_MIN = 5
CAL_EXCLUDE_RANGES = [
    (pd.Timestamp(w['start'], tz='UTC') - pd.Timedelta(minutes=CAL_WINDOW_PAD_MIN),
     pd.Timestamp(w['end'], tz='UTC')   + pd.Timedelta(minutes=CAL_WINDOW_PAD_MIN))
    for wins in WINDOWS_BY_DATE.values() for w in wins
]

def restrict_to_wyo_ambient(series):
    if series is None:
        return None
    s = series[series.index.strftime('%Y%m%d').isin(WYO_DATES)]
    mask = pd.Series(True, index=s.index)
    for t0, t1 in CAL_EXCLUDE_RANGES:
        mask &= ~((s.index >= t0) & (s.index <= t1))
    return s[mask]

U460_AMBIENT    = restrict_to_wyo_ambient(C2H6['Ultra460'])
PICO_AMBIENT    = restrict_to_wyo_ambient(C2H6['Pico017'])
U321_AMBIENT    = restrict_to_wyo_ambient(C2H6['Ultra321'])
CH4_PICARRO_AMB = restrict_to_wyo_ambient(CH4['Picarro'])
C3H8_U321_AMB   = restrict_to_wyo_ambient(C3H8['Ultra321'])

print(f'Ultra460 ambient C2H6 points (WYO days, cal windows excluded): {len(U460_AMBIENT):,}')

In [ ]:
PEAK_HEIGHT_PPB     = 50.0   # minimum Ultra460 C2H6 peak height above baseline
PEAK_PROMINENCE_PPB = 15.0
PEAK_MIN_DISTANCE_S = 30
PEAK_MATCH_WINDOW_S = 10     # +- window around each peak to grab each target's max


def find_peak_matches(ref_series, target_series, height, prominence, min_distance_s, window_s):
    """For each local peak in ref_series (found per UTC day), take each target's max
    within +-window_s seconds of the peak time. One row per matched plume event."""
    records = []
    for day, day_ref in ref_series.groupby(ref_series.index.date):
        if len(day_ref) < 20:
            continue
        peaks, _ = find_peaks(day_ref.values, height=height, prominence=prominence,
                               distance=min_distance_s)
        for p in peaks:
            t_peak = day_ref.index[p]
            row = {'date': str(day), 'peak_time': t_peak, 'ref': float(day_ref.iloc[p])}
            for name, s in target_series.items():
                if s is None:
                    row[name] = np.nan
                    continue
                win = s[t_peak - pd.Timedelta(seconds=window_s): t_peak + pd.Timedelta(seconds=window_s)]
                row[name] = float(win.max()) if len(win) else np.nan
            records.append(row)
    return pd.DataFrame(records)


peaks_df = find_peak_matches(
    U460_AMBIENT,
    {'Pico017': PICO_AMBIENT, 'Ultra321': U321_AMBIENT,
     'CH4_picarro': CH4_PICARRO_AMB, 'C3H8_ultra321': C3H8_U321_AMB},
    height=PEAK_HEIGHT_PPB, prominence=PEAK_PROMINENCE_PPB,
    min_distance_s=PEAK_MIN_DISTANCE_S, window_s=PEAK_MATCH_WINDOW_S,
)
print(f'Plume peaks found: {len(peaks_df)}')
if len(peaks_df):
    print(peaks_df.groupby('date').size().rename('n_peaks').to_string())

In [ ]:
C2H6_COEFS = {}
for inst in ['Pico017', 'Ultra321']:
    sub = peaks_df.dropna(subset=['ref', inst])
    sl, ic, r2, n = linreg(sub['ref'].values, sub[inst].values)
    C2H6_COEFS[inst] = {'slope': sl, 'intercept': ic, 'r2': r2, 'n': n}
    print(f'{inst}:  slope={sl:.4f}  intercept={ic:+.2f} ppb  R2={r2:.4f}  n={n}')

fig = go.Figure()
colors = {'Pico017': '#d62728', 'Ultra321': '#2ca02c'}
for inst in ['Pico017', 'Ultra321']:
    sub = peaks_df.dropna(subset=['ref', inst])
    fig.add_trace(go.Scatter(
        x=sub['ref'], y=sub[inst], mode='markers',
        marker=dict(color=colors[inst], size=6, opacity=0.6), name=inst,
    ))
    c = C2H6_COEFS[inst]
    if np.isfinite(c['slope']):
        xfit = np.array([0, peaks_df['ref'].max() * 1.05])
        fig.add_trace(go.Scatter(x=xfit, y=c['slope'] * xfit + c['intercept'], mode='lines',
                                  line=dict(color=colors[inst], width=2), showlegend=False))
fig.update_layout(title='C2H6 peak-alignment vs Ultra460 (FLAGGED — low confidence, see intro)',
                   xaxis_title='Ultra460 C2H6 peak (ppb)', yaxis_title='Instrument C2H6 peak (ppb)',
                   template='plotly_white', height=480)
fig.show()

# ── Interference diagnostic ─────────────────────────────────────────────────────
# A real spectral cross-sensitivity (CH4 or C3H8 leaking into the C2H6 channel) would
# show residuals trending with CH4/C3H8 level, not just scattering around zero.
print('\nInterference check (|corr| > 0.3 flagged):')
for inst in ['Pico017', 'Ultra321']:
    c = C2H6_COEFS[inst]
    sub = peaks_df.dropna(subset=['ref', inst])
    resid = sub[inst] - (c['slope'] * sub['ref'] + c['intercept'])
    for diag_col in ['CH4_picarro', 'C3H8_ultra321']:
        valid = sub[diag_col].notna() & resid.notna()
        if valid.sum() > 5:
            corr = np.corrcoef(sub.loc[valid, diag_col], resid[valid])[0, 1]
            flag = '  [POSSIBLE INTERFERENCE]' if abs(corr) > 0.3 else ''
            print(f'  {inst:10s} residual vs {diag_col:14s}  corr={corr:+.3f}  (n={int(valid.sum())}){flag}')

---
## E — Assemble and save `calibration_coefs.json`

In [ ]:
def git_info():
    try:
        h = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(REPO_ROOT), text=True).strip()
        dirty = subprocess.call(['git', 'diff', '--quiet'], cwd=str(REPO_ROOT)) != 0
        return h, dirty
    except Exception:
        return 'unknown', False

git_hash, git_dirty = git_info()

corrections = []
for inst, c in CH4_COEFS.items():
    if c is None:
        continue
    corrections.append({
        'gas': 'CH4', 'instrument': inst, 'col_in': 'CH4_ppm', 'col_out': 'CH4_ppm_cal',
        'scale_in': 1.0, 'method': 'tank_multipoint', 'confidence': 'high',
        'cal_date': CAL_DATE_CANONICAL,
        'slope': c['slope'], 'intercept': c['intercept'], 'r2': c['r2'], 'n_points': c['n'],
    })
for inst, c in C3H8_COEFS.items():
    if c is None:
        continue
    corrections.append({
        'gas': 'C3H8', 'instrument': inst, 'col_in': 'C3H8_ppm', 'col_out': 'C3H8_ppm_cal',
        'scale_in': 1.0, 'method': 'tank_multipoint', 'confidence': 'high',
        'cal_date': CAL_DATE_CANONICAL,
        'slope': c['slope'], 'intercept': c['intercept'], 'r2': c['r2'], 'n_points': c['n'],
    })
for inst, c in C2H6_COEFS.items():
    col_in   = 'C2H6_ppb' if inst == 'Pico017' else 'C2H6_ppm'
    scale_in = 1.0 if inst == 'Pico017' else 1000.0
    corrections.append({
        'gas': 'C2H6', 'instrument': inst, 'col_in': col_in, 'col_out': 'C2H6_ppb_cal',
        'scale_in': scale_in, 'method': 'peak_alignment_vs_ultra460', 'confidence': 'low',
        'note': ('Ultra460 used as reference instrument, not independently tank-validated. '
                 'NOAA tank C2H6 (1.63 ppb) is far below ambient plume range, so a tank-anchored '
                 'fit was not usable for this species.'),
        'reference_instrument': 'Ultra460', 'n_peaks': c['n'],
        'slope': c['slope'], 'intercept': c['intercept'], 'r2': c['r2'],
    })

coefs_out = {
    'metadata': {
        'generated_utc': datetime.now(timezone.utc).isoformat(),
        'git_hash': git_hash,
        'git_dirty': git_dirty,
        'formula': 'calibrated = (measured * scale_in - intercept) / slope',
        'cal_date_canonical': CAL_DATE_CANONICAL,
        'drift_qc_dates': sorted(WINDOWS_BY_DATE),
        'ultra460_c2h6_note': 'Ultra460 is the C2H6 reference instrument — passed through uncalibrated.',
    },
    'corrections': corrections,
}

STAGE_04_DIR.mkdir(parents=True, exist_ok=True)
coefs_path = STAGE_04_DIR / 'calibration_coefs.json'
with open(coefs_path, 'w') as f:
    json.dump(coefs_out, f, indent=2)
print(f'Saved {len(corrections)} corrections -> {coefs_path}')

---
## F — Apply calibration → `04_calibrated/`

Reads every good (non-`bad`, non-`bad_timestamp`) Stage 03 aligned file for the four gas
analyzers — `Raw` and `Eng` subdirs only (both carry gas concentration columns; `Spectra`
does not) — applies the relevant correction(s), and writes calibrated Parquet to
`04_calibrated/`, mirroring the Stage 03 layout. Safe to re-run.

In [ ]:
def apply_linear(series, c):
    scale = c.get('scale_in', 1.0)
    return (series * scale - c['intercept']) / c['slope']


CORR_BY_INST = {}
for c in corrections:
    CORR_BY_INST.setdefault(c['instrument'], {})[c['gas']] = c

APPLY_SUBDIRS = {
    'Picarro':  ('WYO_picarro', ['']),
    'Ultra460': ('WYO_aerisultra460', ['Raw', 'Eng']),
    'Ultra321': ('LANL_aerisultra321', ['Raw', 'Eng']),
    'Pico017':  ('LANL_aerispico017', ['Raw', 'Eng']),
}

apply_stats = {}
for inst, (inst_dir, subdirs) in APPLY_SUBDIRS.items():
    inst_corrs = CORR_BY_INST.get(inst, {})
    n_files = n_rows = 0
    for subdir in subdirs:
        src_dir = STAGE_03_DIR / inst_dir / subdir if subdir else STAGE_03_DIR / inst_dir
        if not src_dir.exists():
            continue
        dst_dir = STAGE_04_DIR / inst_dir / subdir if subdir else STAGE_04_DIR / inst_dir
        dst_dir.mkdir(parents=True, exist_ok=True)
        for f in sorted(src_dir.glob('*.parquet')):   # direct children only — skip bad/, bad_timestamp/
            df = pd.read_parquet(f)
            n_added = 0
            for gas, c in inst_corrs.items():
                if c['col_in'] not in df.columns:
                    continue
                df[c['col_out']] = apply_linear(df[c['col_in']], c)
                n_added += 1
            df['cal_coefs_ref'] = 'calibration_coefs.json'
            df.to_parquet(dst_dir / f.name)
            n_files += 1
            n_rows  += len(df)
    apply_stats[inst] = {'files': n_files, 'rows': n_rows}
    print(f'{inst:10s}  {n_files:>4} files  {n_rows:>10,} rows  -> {STAGE_04_DIR / inst_dir}')

apply_manifest = {
    'stage': '04_apply_calibration',
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'git_hash': git_hash,
    'git_dirty': git_dirty,
    'coefs_source': str(coefs_path),
    'instruments': apply_stats,
}
with open(STAGE_04_DIR / 'apply_manifest.json', 'w') as f:
    json.dump(apply_manifest, f, indent=2)
print(f'\nStage 04 apply complete -> {STAGE_04_DIR}')

---
## G — Passthrough: everything else, unchanged

Makes `04_calibrated/` the single complete directory. Straight file copies (no
recomputation needed) of Stage 03 good data that calibration doesn't touch:
Spectra/Spectralite for the three Aeris instruments (no concentration columns), and
GPS/Anem/Sprinter/LGR in full (no tank or cross-cal coverage exists for any of them —
LGR wasn't deployed until Mar 10, after all three cal events).

In [ ]:
import shutil

def copy_passthrough(inst_dir, subdir, label):
    src_dir = STAGE_03_DIR / inst_dir / subdir if subdir else STAGE_03_DIR / inst_dir
    if not src_dir.exists():
        print(f'{label:30s}  (no Stage 03 data)')
        return 0
    dst_dir = STAGE_04_DIR / inst_dir / subdir if subdir else STAGE_04_DIR / inst_dir
    dst_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in sorted(src_dir.glob('*.parquet')):   # direct children only — skip bad/, bad_timestamp/
        shutil.copy2(f, dst_dir / f.name)
        n += 1
    print(f'{label:30s}  {n:>4} files  -> {dst_dir}')
    return n


passthrough_stats = {}

# ── Spectra/Spectralite: same 3 Aeris instruments, no concentration columns ────────
SPECTRA_SUBDIR = {
    'Ultra460': ('WYO_aerisultra460', 'Spectralite'),
    'Ultra321': ('LANL_aerisultra321', 'Spectra'),
    'Pico017':  ('LANL_aerispico017', 'Spectra'),
}
for inst, (inst_dir, subdir) in SPECTRA_SUBDIR.items():
    passthrough_stats[f'{inst_dir}/{subdir}'] = copy_passthrough(inst_dir, subdir, f'{inst}/{subdir}')

# ── Instruments with no calibration treatment at all: straight copy ───────────────
PASSTHROUGH_ONLY = ['LANL_GPS', 'LANL_Anem', 'WYO_sprinter', 'UOU_LGR']
for inst_dir in PASSTHROUGH_ONLY:
    passthrough_stats[inst_dir] = copy_passthrough(inst_dir, '', inst_dir)

with open(STAGE_04_DIR / 'passthrough_manifest.json', 'w') as f:
    json.dump({
        'stage': '04_passthrough',
        'run_utc': datetime.now(timezone.utc).isoformat(),
        'note': ('Straight copies of Stage 03 good data with no calibration applied — '
                 'either no concentration columns (Spectra) or no tank/cross-cal coverage '
                 '(GPS, Anem, Sprinter, LGR). No cal_coefs_ref column added, which is how '
                 'to tell these apart from calibrated files.'),
        'copied': passthrough_stats,
    }, f, indent=2)
print(f'\nStage 04 passthrough complete -> {STAGE_04_DIR}')